# Modified Newton method - Thermal explosion

In [ ]:
#    APM41012EP course notebook - Chapter 6 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Modified Newton on the thermal explosion
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np

from scipy.sparse import diags
from scipy.sparse.linalg import spsolve, eigs, factorized

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Problem formulation

We want to solve the elliptic problem given by the Poisson equation subject to Dirichlet boundary conditions and to a nonlinear source term of thermal explosion type:

$$
\left\{
\begin{aligned}
-\mathrm{d}_x^2 \theta(x) & = \lambda_{\mathrm{FK}} \exp(\theta(x)) &&x\in \Omega = ]0;2[, \\
        \theta(x) & =  0 && x\in \{0,2\},
\end{aligned}
\right.
$$

where $\lambda_{\mathrm{FK}}$ is the Frank-Kamenetskii parameter. 

In [ ]:
class thermal_explosion_model:
    
    def __init__(self, lamb, xmin, xmax, nx):
        self.lamb = lamb
        self.xmin = xmin
        self.xmax = xmax
        self.nx = nx
        self.dx = (xmax-xmin)/(nx+1)

    def fcn(self, theta):
        lamb = self.lamb
        nx = self.nx
        dx = self.dx
        oneoverdxdx = 1/dx**2
            
        lap = np.zeros(nx)
        lap[0]    = oneoverdxdx * (2*theta[0] - theta[1])
        lap[1:-1] = oneoverdxdx * (-theta[:-2] + 2*theta[1:-1] - theta[2:])
        lap[-1]   = oneoverdxdx * (-theta[-2] + 2*theta[-1])

        return lap - lamb*np.exp(theta)
    
    def jac(self, theta):
        lamb = self.lamb
        nx = self.nx
        dx = self.dx
        diagonals = [np.repeat(2/dx**2, nx) - lamb*np.exp(theta), np.repeat(-1/dx**2, nx-1), np.repeat(-1/dx**2, nx-1)]
        return diags(diagonals, [0, -1, 1])

In [ ]:
def newton(f, jac, x0, tol=1.e-10, max_iter=50, verbose=False):
    
    xk = np.copy(x0)
    
    res = np.zeros(max_iter+1)
    res[0] = np.linalg.norm(f(xk))
    
    incre = np.zeros(max_iter+1)
    incre[0] = 0
    
    # Newton iteration        
    for k in range(max_iter):
        increment = spsolve(jac(xk).tocsr(), -f(xk)) 
        xk = xk + increment
        incre[k+1] = np.linalg.norm(increment)/np.linalg.norm(xk)
        reskp1 = np.linalg.norm(f(xk))
        if (verbose): print(f"Iteration nb {k+1:3d}: ||f(xk)|| = {reskp1:14.8e}")
        res[k+1] = reskp1
        if ( np.linalg.norm(f(xk)) < tol ): break
 
    return xk, res[:k+2], incre[:k+2]

def modified_newton(f, jac, x0, tol=1.e-10, max_iter=50, freq=2, verbose=False):
    
    xk = np.copy(x0)
    
    res = np.zeros(max_iter+1)
    res[0] = np.linalg.norm(f(xk))
    
    incre = np.zeros(max_iter+1)
    incre[0] = 0
    
    # Newton iteration        
    for k in range(max_iter):
        if k%freq == 0:
            jac_mat = jac(xk)  
            solve = factorized(jac_mat.tocsc())
        increment = solve(-f(xk)) 
        xk = xk + increment
        incre[k+1] = np.linalg.norm(increment)/np.linalg.norm(xk)
        reskp1 = np.linalg.norm(f(xk))
        if (verbose): print(f"Iteration nb {k+1:3d}: ||f(xk)|| = {reskp1:14.8e}")
        res[k+1] = reskp1
        if ( np.linalg.norm(f(xk)) < tol ): break
    
    return xk, res[:k+2], incre[:k+2]

## Modified Newton method with periodic re-evaluation of the Jacobian 

The aim of this second experiment is to solve the previous problem, but while limiting the computational cost of evaluating the Jacobian and of its decomposition, by doing so only periodically, or even only once at the beginning of the computation (the parameter "freq" is the re-evaluation frequency of the Jacobian matrix, which we set to max_iter if we only want a single evaluation at the beginning of the computation).

The computation should first be carried out with $\lambda_{\mathrm{FK}}=0.1$, where we observe that a single evaluation at the beginning of the computation is enough to converge, because the problem is relatively weakly nonlinear and reasonably ill-conditioned. On the other hand, this becomes impossible for $\lambda_{\mathrm{FK}}=0.878$, close to the limit point, because of the strong nonlinearity of the system and of the sharp increase of the condition number.

In [ ]:
# Limit value of lambda = 0.88 
lamb = 0.1
xmin = 0.
xmax = 2.
# nb of points including boundary conditions
nxib = 1002
nx = nxib-2

theta_ini = np.zeros(nx)

# Limit value of lambda = 0.88 
lamb = 0.1
print('***************************************')
print(f"Model for lambda: {lamb}")

tem = thermal_explosion_model(lamb=lamb, xmin=xmin, xmax=xmax, nx=nx)
fcn = tem.fcn
jac = tem.jac

print(f"\nNewton algorithm")
max_iter = 20
theta_sol, res, increment = newton(fcn, jac, theta_ini, tol=1.e-14, max_iter=max_iter, verbose=True)

print(f"\nModified Newton algorithm")
# here we set the evaluation frequency of the Jacobian and of its decomposition
# if freq = max_iter then only one evaluation of the Jacobian is performed at the beginning
freq = 3
theta_sol_mod, res_mod, increment_mod = modified_newton(fcn, jac, theta_ini, tol=1.e-14, max_iter=max_iter, freq=freq, verbose=True)

jac_eq = jac(theta_sol)

eig_val_min = np.real(eigs(jac_eq, k=1, which='SR')[0])[0]
eig_val_max = np.real(eigs(jac_eq, k=1, which='LR')[0])[0]
print(f"\nCondition number of the Jacobian matrix at equilibrium: {eig_val_max/eig_val_min}")
print(f"\nCondition number of the Laplacian matrix: {(2*(nx+1)/np.pi)**2}")

In [ ]:
dx = (xmax-xmin)/(nxib-1)
x = np.linspace(0+dx, 1-dx, nx)

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12, 
                    subplot_titles=("Evolution of the residual", "Evolution of the increment"))
fig.add_trace(go.Scatter(x=np.arange(res.size), y=res, mode='lines+markers', name='Newton',
                         line=dict(dash='dot'), legendgroup = '1'), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(res_mod.size), y=res_mod, mode='lines+markers', name='Modified Newton',
                        line=dict(dash='dot'), legendgroup = '1'), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(1,increment.size), y=increment[1:], mode='lines+markers', name='Newton',
                         line=dict(color='rgb(76,114,176)', dash='dot'), legendgroup = '2'), row=2, col=1)
fig.add_trace(go.Scatter(x=np.arange(1,increment_mod.size), y=increment_mod[1:], mode='lines+markers', name='Modified Newton',
                         line=dict(color='rgb(221,132,82)', dash='dot'), legendgroup = '2'), row=2, col=1)

freq = 3

#create slider
steps = []
for lamb_i in [0.1, 0.5, 0.85, 0.878]:
    tem = thermal_explosion_model(lamb=lamb_i, xmin=xmin, xmax=xmax, nx=nx)
    fcn = tem.fcn
    jac = tem.jac
    theta_sol, res, increment = newton(fcn, jac, theta_ini, tol=1.e-13, max_iter=max_iter, verbose=False)    
    theta_sol_mod, res_mod, increment_mod = modified_newton(fcn, jac, theta_ini, tol=1.e-13, max_iter=max_iter, freq=freq, verbose=False)
    step = dict(method="update", label = f"{lamb_i:.3f}", 
                args=[{"x": [np.arange(res.size),np.arange(res_mod.size),np.arange(1,increment.size),np.arange(1,increment_mod.size)], 
                       "y": [res, res_mod, increment[1:], increment_mod[1:]]}])
    steps.append(step)
sliders = [dict(currentvalue={'prefix': 'lambda = '}, steps=steps)]


fig.update_xaxes(range=[-0.5,increment.size-0.5], row=1)    
fig.update_yaxes(type="log", exponentformat='e', row=1)   
fig.update_xaxes(range=[-0.5,increment.size-0.5], row=2)    
fig.update_yaxes(type="log", exponentformat='e', row=2)    
fig.update_layout(sliders=sliders, height=1000, legend_tracegroupgap=430, legend=dict(x=0.77, bgcolor='rgba(0,0,0,0)'))
fig.show()